# Phase A — Leave-One-VIB-Out on WavLM-Base-Plus

**Goal:** determine which group's VIB carries the gain in `per_group_vib` (11.67% ITW EER on Base-Plus). Three variants, each disabling exactly one group's VIB while keeping the other two active. RawBoost on (matching the 11.67% reference).

**Variants:**
1. `drop_vib_g0` — disable VIB on Group 0 (early/acoustic layers, ~layers 0-3 of WavLM-Base-Plus)
2. `drop_vib_g1` — disable VIB on Group 1 (middle/phonetic layers, ~4-7)
3. `drop_vib_g2` — disable VIB on Group 2 (late/linguistic layers, ~8-11)

**'Disabled' semantics:** the VIB encoder still runs (so the 256-dim projection feeds the downstream concat normally), but `z_k = mu_k` (deterministic — no reparam noise at training time) and `kl_k = 0` (no KL pressure on that group). The other groups' VIBs run normally.

**Decision rule:**
- Any drop_vib_gK at < 11.67% → that group's VIB was hurting; the winner becomes the Base-Plus best.
- All three drop_vib_gK at > 11.67% → every group's VIB contributes; per_group_vib stays as-is.
- One drop_vib_gK ≪ 11.67% (e.g., < 10%) → strong leave-one-out winner; informs Phase B scaled-beta config.

Self-contained: Run All from scratch. Does not depend on phase_vib_stagger or other notebooks.


## 1. Environment Setup


In [ ]:
!pip install -q transformers soundfile librosa scikit-learn tqdm safetensors

from google.colab import drive
drive.mount('/content/drive')

import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name} ({gpu.total_memory / 1e9:.1f} GB)')
else:
    raise RuntimeError('No GPU detected')


Mounted at /content/drive
GPU: NVIDIA A100-SXM4-40GB (42.4 GB)


## 2. Copy GLADA Package to Local Runtime


In [ ]:
import shutil, os, sys

CANDIDATES = [
    "/content/drive/Othercomputers/Jakob's MacBook Air/Thesis/298-deliverables/glada-code/glada",
    '/content/drive/MyDrive/Jakob/glada_pkg/glada',
    '/content/drive/MyDrive/Jakob/glada_pkg',
    '/content/drive/MyDrive/Jakob/glada',
]
DRIVE_PKG = next((c for c in CANDIDATES if os.path.exists(os.path.join(c, '__init__.py'))), None)
if DRIVE_PKG is None:
    raise FileNotFoundError(
        'No glada/__init__.py found in candidates. Run:\n'
        "  !find /content/drive -name '__init__.py' -path '*/glada/*' 2>/dev/null | head\n"
        'and add the parent directory to CANDIDATES above.'
    )

LOCAL_PKG = '/content/glada'
if os.path.exists(LOCAL_PKG): shutil.rmtree(LOCAL_PKG)
shutil.copytree(DRIVE_PKG, LOCAL_PKG)

for mod in list(sys.modules):
    if mod == 'glada' or mod.startswith('glada.'): del sys.modules[mod]
sys.path = [p for p in sys.path if 'glada' not in p]
if '/content' not in sys.path: sys.path.insert(0, '/content')
print(f'Package source: {DRIVE_PKG}\nCopied to:      {LOCAL_PKG}')


Package source: /content/drive/MyDrive/Jakob/glada_pkg/glada
Copied to:      /content/glada


## 3. Datasets (ASVspoof 2019 LA + ITW)


In [ ]:
import tarfile

def extract_dataset(tar_path, extract_to, name):
    if os.path.exists(extract_to) and len(os.listdir(extract_to)) > 0:
        print(f'{name}: already extracted to {extract_to}'); return
    os.makedirs(extract_to, exist_ok=True)
    print(f'{name}: extracting {tar_path} -> {extract_to} ...')
    with tarfile.open(tar_path) as t: t.extractall(extract_to)
    print(f'{name}: done.')

extract_dataset('/content/drive/MyDrive/Jakob/Datasets/LA.tar', '/content/data/asvspoof', 'ASVspoof 2019 LA')
extract_dataset('/content/drive/MyDrive/Jakob/Datasets/release_in_the_wild.tar', '/content/data/itw', 'In-the-Wild')

ASV_PROTO      = '/content/data/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt'
ASV_AUDIO      = '/content/data/asvspoof/LA/ASVspoof2019_LA_train/flac'
ASV_EVAL_PROTO = '/content/data/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt'
ASV_EVAL_AUDIO = '/content/data/asvspoof/LA/ASVspoof2019_LA_eval/flac'
ITW_META       = '/content/data/itw/release_in_the_wild/meta.csv'
ITW_AUDIO      = '/content/data/itw/release_in_the_wild'
for p in [ASV_PROTO, ASV_AUDIO, ASV_EVAL_PROTO, ITW_META]:
    assert os.path.exists(p), f'Missing: {p}'
print('Datasets verified.')


ASVspoof 2019 LA: already extracted to /content/data/asvspoof
In-the-Wild: extracting /content/drive/MyDrive/Jakob/Datasets/release_in_the_wild.tar -> /content/data/itw ...


/tmp/ipykernel_692/3241063313.py:8: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  with tarfile.open(tar_path) as t: t.extractall(extract_to)


In-the-Wild: done.
Datasets verified.


## 4. Imports


In [ ]:
from glada import GLADAConfig, BACKBONE_REGISTRY, set_seed
from glada.datasets import (
    ASVspoofDatasetWithSpeaker,
    ASVspoofEvalDataset,
    InTheWildDataset,
    create_dataloaders,
)
from glada.augmentation import RawBoost
from glada.models import GLADAModel
from glada.models.vib import VIBBottleneck
from glada.models.grl import GradientReversal
from glada.models.discriminators import GroupedLayerDomainDiscriminators
from glada.trainers import GLADATrainer

import copy, gc, json, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

print('Imports OK')


Imports OK


## 5. INLINE: `PerGroupVIBModel` (with leave-one-out + per-group beta flags)

Subclasses `GLADAModel`. Adds K `VIBBottleneck` modules (one per group), each compressing a `(B, 2D)` group ASP output to `(B, bottleneck_dim)`. The per-group discriminator then receives the compressed `z_k` rather than the raw `g_k`.

**New flags vs. the phase_vib_stagger version:**
- `vib_disabled_groups: tuple = ()` — for these group indices, `z_k = mu_k` (deterministic, no reparam sampling) and `kl_k = 0`. Used by `phase_a_leaveout` to ablate one group's compression at a time.
- `vib_betas: tuple = None` — per-group beta override. If provided, `kl_total = sum_k vib_betas[k] * kl_k` instead of the default uniform mean. Used by `phase_b_confirm` to apply scaled betas based on Phase A leave-one-out results.

Backwards compatibility: with no extra kwargs, behavior matches the original phase_vib_stagger inline class.


In [ ]:
class PerGroupVIBModel(GLADAModel):
    """GLADAModel with one VIBBottleneck per group, applied BEFORE the
    per-group discriminators. The hierarchical pool + global VIB are
    replaced by a simple linear projection over the concatenated
    per-group compressed representations.

    Args (in addition to GLADAConfig):
        vib_disabled_groups: tuple of group indices to disable VIB on
            (z_k = mu_k, kl_k = 0). Used for leave-one-out ablation.
        vib_betas: tuple of length K with per-group beta multipliers.
            None -> uniform mean (original behavior). Tuple -> sum of
            beta-weighted KLs.
    """

    def __init__(self, config, vib_disabled_groups=(), vib_betas=None):
        super().__init__(config)
        K = config.num_groups
        D = config.hidden_dim
        bd = config.bottleneck_dim
        self.group_vibs = nn.ModuleList([
            VIBBottleneck(2 * D, bd) for _ in range(K)
        ])
        self.group_compressed_proj = nn.Linear(K * bd, bd)
        if self.glada_disc is not None:
            self.glada_disc = GroupedLayerDomainDiscriminators(
                num_groups=K,
                input_dim=bd,
                hidden_dim=config.glada_disc_hidden,
            )

        # Phase A controls.
        for k in vib_disabled_groups:
            assert 0 <= k < K, f'vib_disabled_groups index {k} not in [0,{K})'
        self._vib_disabled = set(vib_disabled_groups)
        if vib_betas is not None:
            assert len(vib_betas) == K, f'vib_betas must have len {K}, got {len(vib_betas)}'
            self.register_buffer('_vib_betas',
                torch.tensor(vib_betas, dtype=torch.float32))
        else:
            self._vib_betas = None

    def forward(self, input_values, attention_mask=None):
        backbone_frozen = not any(p.requires_grad for p in self.backbone.parameters())
        if backbone_frozen:
            with torch.no_grad():
                outputs = self.backbone(input_values, attention_mask=attention_mask, output_hidden_states=True)
        else:
            outputs = self.backbone(input_values, attention_mask=attention_mask, output_hidden_states=True)
        hidden_states = list(outputs.hidden_states)

        if attention_mask is not None:
            input_lengths = attention_mask.sum(dim=1)
            output_lengths = ((input_lengths - 1) // 320) + 1
            T = hidden_states[0].shape[1]
            frame_mask = (torch.arange(T, device=attention_mask.device).unsqueeze(0) < output_lengths.unsqueeze(1))
        else:
            frame_mask = None

        hidden_states = self.layer_norms(hidden_states)
        _, cka_loss, group_outputs = self.pooling(hidden_states, frame_mask=frame_mask)
        if not self.config.use_cka_loss:
            cka_loss = torch.tensor(0.0, device=group_outputs[0].device)

        # Per-group VIB with leave-one-out + per-group beta support.
        z_groups = []
        kl_per_group = []
        for k, g_k in enumerate(group_outputs):
            if k in self._vib_disabled:
                # Disabled: deterministic z_k = mu_k, kl_k = 0.
                z_k, _ = self.group_vibs[k](g_k, training=False)
                kl_k = torch.zeros((), device=g_k.device)
            else:
                z_k, kl_k = self.group_vibs[k](g_k, training=self.training)
            z_groups.append(z_k)
            kl_per_group.append(kl_k)

        # Aggregate KL across groups.
        if self._vib_betas is not None:
            # Beta-weighted sum (per-group beta).
            kl_total = sum(self._vib_betas[k] * kl_per_group[k] for k in range(len(group_outputs)))
        else:
            # Uniform mean (original behavior; effective per-group beta = config.vib_beta / G).
            kl_total = sum(kl_per_group) / max(len(group_outputs), 1)

        z_concat = torch.cat(z_groups, dim=-1)
        z = self.group_compressed_proj(z_concat)

        spoof_logits = self.task_classifier(z)
        result = {
            'spoof_logits': spoof_logits,
            'kl_loss': kl_total,
            'cka_loss': cka_loss,
            'features': z,
        }

        if self.glada_disc is not None:
            result['glada_domain_logits'] = self.glada_disc(z_groups, self.domain_grl_alpha)
            result['group_outputs'] = z_groups

        if self.domain_disc is not None:
            z_domain = GradientReversal.apply(z, self.domain_grl_alpha)
            result['domain_logits'] = self.domain_disc(z_domain)

        if self.speaker_adversary is not None:
            z_speaker = GradientReversal.apply(z, self.speaker_grl_alpha)
            result['speaker_logits'] = self.speaker_adversary(z_speaker)

        return result

print('PerGroupVIBModel defined (with vib_disabled_groups + vib_betas).')


PerGroupVIBModel defined (with vib_disabled_groups + vib_betas).


## 7. Configuration Helper


In [ ]:
USE_FULL_EPOCHS = False
PHASE1 = 5  if USE_FULL_EPOCHS else 3
PHASE2 = 45 if USE_FULL_EPOCHS else 17

DRIVE_LOG_DIR = '/content/drive/MyDrive/Jakob/Experiments/phase_a_leaveout'
os.makedirs(DRIVE_LOG_DIR, exist_ok=True)

def make_config(name, backbone='wavlm-base-plus', seed=42):
    cfg = GLADAConfig(
        experiment_name=name,
        backbone=backbone,
        seed=seed,
        batch_size=64,  # FIXED for apples-to-apples vs 11.67% reference
        num_workers=0,
        phase1_epochs=PHASE1,
        phase2_epochs=PHASE2,
        drive_log_dir=DRIVE_LOG_DIR,
        run_id=time.strftime('%Y%m%d_%H%M%S'),
    )
    return cfg


## 8. Datasets (RawBoost=ON)


In [ ]:
base_for_data = make_config('_data_setup')

source_ds = ASVspoofDatasetWithSpeaker(
    protocol_path=ASV_PROTO,
    audio_dir=ASV_AUDIO,
    config=base_for_data,
    augmenter=RawBoost,  # RawBoost only — matches phase_vib_stagger reference
)
target_ds = InTheWildDataset(
    meta_csv_path=ITW_META, audio_dir=ITW_AUDIO,
    config=base_for_data, use_labels=False,
)
eval_ds = InTheWildDataset(
    meta_csv_path=ITW_META, audio_dir=ITW_AUDIO,
    config=base_for_data, use_labels=True,
)
extra_eval = {
    '2019_LA': ASVspoofEvalDataset(
        protocol_path=ASV_EVAL_PROTO, audio_dir=ASV_EVAL_AUDIO, config=base_for_data,
    ),
}
num_speakers = len(source_ds.speaker_to_id) if hasattr(source_ds, 'speaker_to_id') else None
print(f'Source: {len(source_ds):,} ({num_speakers} speakers)')
print(f'Target: {len(target_ds):,}, Eval: {len(eval_ds):,}, 2019 LA: {len(extra_eval["2019_LA"]):,}')


  ASVspoof: 25380 samples, 20 speakers, rawboost=0.5
  ITW: 31779 samples (unlabeled returned)
  ITW: 31779 samples (labeled returned)
  ASVspoof Eval: 71237 samples
Source: 25,380 (20 speakers)
Target: 31,779, Eval: 31,779, 2019 LA: 71,237


## 8b. Variant Definitions

Each variant is `per_group_vib` with `vib_disabled_groups=(k,)` for k in {0,1,2}. Other model knobs (config_overrides) are unchanged from the 11.67% reference.


In [ ]:
RAWBOOST_DEFAULT = True  # all variants use RawBoost (matches 11.67% reference)
RESULTS_FILENAME = 'phase_a_leaveout_results.json'

# DIAGNOSTIC RUN ORDER: 'sanity_no_disable' first to confirm the new flag plumbing
# didn't break anything (it should reproduce the 11.67% reference). Only after that
# trains cleanly should drop_vib_g1 and drop_vib_g2 be uncommented and run.
VARIANTS = {
    # 'sanity_no_disable': {  # control: matches the 11.67% per_group_vib reference
    #     'model_class': 'PerGroupVIBModel',
    #     'model_kwargs': {},
    # },
    'drop_vib_g0': {
        'model_class': 'PerGroupVIBModel',
        'model_kwargs': {'vib_disabled_groups': (0,)},
    },
    'drop_vib_g1': {
        'model_class': 'PerGroupVIBModel',
        'model_kwargs': {'vib_disabled_groups': (1,)},
    },
    'drop_vib_g2': {
        'model_class': 'PerGroupVIBModel',
        'model_kwargs': {'vib_disabled_groups': (2,)},
    },
}
for name, spec in VARIANTS.items():
    print(f'  {name:<20s} kwargs={spec["model_kwargs"]}')


  drop_vib_g0          kwargs={'vib_disabled_groups': (0,)}
  drop_vib_g1          kwargs={'vib_disabled_groups': (1,)}
  drop_vib_g2          kwargs={'vib_disabled_groups': (2,)}


## 8c. Diagnostic Pre-flight (DELETE ONCE STABLE)

Two safeguards to make CUDA crashes legible. Remove these cells once the sweep runs cleanly:

1. **Synchronous CUDA + anomaly detection** so kernel asserts surface as a real Python traceback at the exact failing line. Runs ~20% slower; only keep while debugging.
2. **One-batch sanity probe** on the source loader to catch label-range bugs CPU-side before the GPU assert fires. Verifies `speaker_id ∈ [0, num_speakers)` and `label ∈ {0, 1}`.


In [ ]:
# 1) Synchronous CUDA: turn the async kernel asserts into Python tracebacks.
#    Note: torch.autograd.set_detect_anomaly(True) is OMITTED here -- it retains the
#    full computation graph for every op and OOMs on the unfrozen ~100M backbone.
#    The two env vars below are cheap and only affect error reporting, not memory.
import os
# os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
# os.environ['TORCH_USE_CUDA_DSA'] = '1'
# print('CUDA_LAUNCH_BLOCKING=1, TORCH_USE_CUDA_DSA=1 (anomaly detection OFF -- it OOMs the backbone)')
os.environ.pop('CUDA_LAUNCH_BLOCKING', None)
os.environ.pop('TORCH_USE_CUDA_DSA', None)
print('CLB:', os.environ.get('CUDA_LAUNCH_BLOCKING'), 'DSA:',
os.environ.get('TORCH_USE_CUDA_DSA'))
# 2) One-batch sanity probe.
print('\nSanity-checking source loader (3 batches)...')
sanity_loader = torch.utils.data.DataLoader(source_ds, batch_size=4, shuffle=True, num_workers=0)
for i, batch in enumerate(sanity_loader):
    if i >= 3: break
    spk = batch.get('speaker_id')
    lbl = batch.get('task_label')  # ASVspoofDatasetWithSpeaker returns task_label, not label
    assert lbl is not None, f'No task_label in batch keys: {list(batch.keys())}'
    assert lbl.min() >= 0 and lbl.max() <= 1, f'task_label out of range: [{int(lbl.min())}, {int(lbl.max())}]'
    if spk is not None:
        assert spk.min() >= 0 and spk.max() < num_speakers, \
            f'speaker_id out of range: [{int(spk.min())}, {int(spk.max())}] vs num_speakers={num_speakers}'
    shape_summary = [(k, tuple(v.shape)) for k, v in batch.items() if hasattr(v, 'shape')]
    print(f'  batch {i}: shapes={shape_summary}, task_labels={lbl.unique().tolist()}, '
          f'speakers={spk.unique().tolist() if spk is not None else "n/a"}')
print('Source loader sanity OK.')


CLB: None DSA: None

Sanity-checking source loader (3 batches)...
  batch 0: shapes=[('input_values', (4, 64000)), ('attention_mask', (4, 64000)), ('task_label', (4,)), ('speaker_id', (4,))], task_labels=[0, 1], speakers=[2, 5, 6, 18]
  batch 1: shapes=[('input_values', (4, 64000)), ('attention_mask', (4, 64000)), ('task_label', (4,)), ('speaker_id', (4,))], task_labels=[0], speakers=[5, 9, 13, 17]
  batch 2: shapes=[('input_values', (4, 64000)), ('attention_mask', (4, 64000)), ('task_label', (4,)), ('speaker_id', (4,))], task_labels=[0], speakers=[9, 13, 16, 19]
Source loader sanity OK.


## 9. Custom Sweep Loop

Mirrors `phase_vib_stagger`'s pattern but threads `model_kwargs` through the variant trainer factory so each variant can set `vib_disabled_groups` / `vib_betas` independently. Per-variant config overrides (e.g., backbone, vib_beta) are also supported.


In [ ]:
MODEL_CLASSES = {'GLADAModel': GLADAModel, 'PerGroupVIBModel': PerGroupVIBModel}

all_results = {}
sweep_start = time.time()
results_path = os.path.join(DRIVE_LOG_DIR, RESULTS_FILENAME)

for i, (name, spec) in enumerate(VARIANTS.items()):
    elapsed = (time.time() - sweep_start) / 60
    print(f'\n{"="*60}\nVariant {i+1}/{len(VARIANTS)}: {name}\n  elapsed: {elapsed:.1f} min\n{"="*60}')

    backbone = spec.get('backbone', None)
    cfg_kwargs = dict()
    if backbone is not None:
        cfg_kwargs['backbone'] = backbone
    cfg = make_config(name, **cfg_kwargs)
    for attr, val in spec.get('config_overrides', {}).items():
        setattr(cfg, attr, val)
    set_seed(cfg.seed)

    # If this variant changes RawBoost, rebuild source_ds. Otherwise reuse.
    use_rb = spec.get('use_rawboost', RAWBOOST_DEFAULT)
    if use_rb != RAWBOOST_DEFAULT:
        from glada.augmentation import RawBoost as _RB
        aug = _RB if use_rb else None
        local_source_ds = ASVspoofDatasetWithSpeaker(
            protocol_path=ASV_PROTO, audio_dir=ASV_AUDIO,
            config=cfg, augmenter=aug,
        )
    else:
        local_source_ds = source_ds

    loaders = create_dataloaders(cfg, local_source_ds, target_ds, eval_ds, extra_eval)

    ModelCls = MODEL_CLASSES[spec['model_class']]
    model_kwargs = spec.get('model_kwargs', {})
    TrainerCls = GLADATrainer  # No staggered variants in Phase A/B

    class _VariantTrainer(TrainerCls):
        def __init__(self, *a, **kw):
            super().__init__(*a, **kw)
            old_model = self.model
            new_model = ModelCls(self.config, **model_kwargs).to(self.device)
            old_state = old_model.state_dict()
            new_state = new_model.state_dict()
            for k in new_state.keys():
                if k in old_state and old_state[k].shape == new_state[k].shape:
                    new_state[k] = old_state[k]
            new_model.load_state_dict(new_state, strict=False)
            self.model = new_model
            self._setup_phase1()
            trainable = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
            total = sum(p.numel() for p in self.model.parameters())
            print(f'  [VARIANT] {ModelCls.__name__} kwargs={model_kwargs}; '
                  f'trainable={trainable:,}/{total:,}')

    try:
        trainer_kwargs = dict(
            config=cfg,
            source_loader=loaders['source'],
            target_loader=loaders['target'],
            eval_loader=loaders['eval'],
            extra_eval_loaders={k: v for k, v in loaders.items() if k not in ('source', 'target', 'eval')},
            num_speakers=num_speakers,
        )
        trainer = _VariantTrainer(**trainer_kwargs)
        results = trainer.train()
        all_results[name] = results
    except Exception as e:
        import traceback; traceback.print_exc()
        all_results[name] = {'error': str(e)}

    save = {k: {kk: vv for kk, vv in v.items() if kk != 'history'} for k, v in all_results.items()}
    with open(results_path, 'w') as f:
        json.dump(save, f, indent=2, default=str)

    if 'trainer' in dir(): del trainer
    torch.cuda.empty_cache(); gc.collect()

print(f'\nSweep complete: {len(all_results)} variants in {(time.time()-sweep_start)/60:.1f} min')



Variant 1/3: drop_vib_g0
  elapsed: 0.0 min
  Updating num_speakers: 107 -> 20

Building GLADA model: wavlm-base-plus


Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

  Trainable: 6,356,140 / 100,738,076
  Run dir: /content/drive/MyDrive/Jakob/Experiments/phase_a_leaveout/drop_vib_g0/20260506_202402


Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

  [VARIANT] PerGroupVIBModel kwargs={'vib_disabled_groups': (0,)}; trainable=22,587,308/116,969,244

GLADA Training | wavlm-base-plus | seed=42
  Phase 1: 3 ep | Phase 2: 17 ep
  Loss Weights: D=0.1, KL=0.01, SPK=0.1, CKA=0.01
  KL Annealing: 0 -> 0.01 over 5 epochs
  Phase 2 Adv Ramp: 0.0 -> 1.0 over 3 P2 epochs (+CKA)


Ep 1:   0%|          | 0/396 [00:00<?, ?it/s]

... (131 outputs left out)

/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


  Ep 4 P2: L=0.092 KL=1.05 | ITW=17.43% | 2019_LA=1.35%


Ep 5:   0%|          | 0/396 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


## 10. Decision Table

Negative delta vs `per_group_vib (BP)` (11.67%) means the leave-one-out variant **beats** the reference. The smallest (most negative) delta picks the winner.


In [ ]:
REFERENCE = {
    'full_glada':              19.71,
    'no_glada':                14.89,
    'no_cka':                  12.98,
    'per_group_vib (BP)':      11.67,  # the 11.67% reference
    'bb_wavlm-large_s42':      12.64,
    'per_group_vib (Large)':   13.52,
}

header = f'{"variant":<32s} {"ITW EER":>10s}' + ''.join(f'  vs {k:<18s}' for k in REFERENCE.keys())
print(header); print('-' * len(header))
for name, res in all_results.items():
    if 'error' in res:
        print(f'{name:<32s} FAILED: {res["error"][:60]}'); continue
    eer = res.get('best_eer') or res.get('eval_metrics', {}).get('ITW', {}).get('eer')
    if eer is None:
        print(f'{name:<32s} NO EER'); continue
    deltas = ''.join(f'  {(eer - v):>+18.2f}' for v in REFERENCE.values())
    print(f'{name:<32s} {eer:>10.2f}{deltas}')

print()
print('Reference (no retraining):')
for name, eer in REFERENCE.items():
    print(f'  {name:<32s} {eer:>10.2f}')


## 11. Verdict & Next Step

If a variant beats 11.67% by more than 0.5 pp (significantly outside the bootstrap CI of [11.25, 12.10] for the reference): that group's VIB was actively hurting. The winner config (with that group's VIB disabled) becomes the Base-Plus best, and Phase B's scaled-beta variant is built around it (set the disabled group's β to 0 or near-0, redistribute to the other groups).

If all three variants are ≥ 11.67%: every group's VIB contributes; per_group_vib stays as-is, and Phase B focuses on per-group β tuning (e.g., scale up the most informative group's β) rather than dropping anything.

If results are mixed and noisy (within ±0.3 pp): re-run with seed 123 to disambiguate before authoring Phase B.


In [ ]:
valid = {n: r for n, r in all_results.items() if 'error' not in r and r.get('best_eer') is not None}
if not valid:
    print('All variants failed; cannot pick a winner.'); raise SystemExit(0)
BP_REFERENCE = 11.67
best_name = min(valid, key=lambda n: valid[n]['best_eer'])
best_eer = valid[best_name]['best_eer']
delta = best_eer - BP_REFERENCE
print()
print('=' * 70)
print(f'Winner: {best_name} at {best_eer:.2f}% (delta vs 11.67% = {delta:+.2f} pp)')
if delta < -0.5:
    print('  -> SIGNIFICANT improvement; this group VIB was hurting.')
    print(f'  -> Phase B: build scaled-beta variant with vib_disabled_groups={best_name[-2:]} or beta_k = 0.')
elif delta < 0:
    print('  -> Marginal improvement; check seed sensitivity before claiming.')
else:
    print('  -> No leave-one-out beat 11.67%; every group VIB contributes.')
    print('  -> Phase B: focus on per-group beta tuning instead of disabling.')
print('=' * 70)
